---
sidebar_position: 8
title: ML Feature Pipeline
---

# ML Feature Pipeline

`varistar.ml` turns a collection of `(TimeSeries, LightCurve)` pairs into a
feature matrix, then clusters it. `Dataset` is the interactive entry point;
`FeaturePipeline` is the CLI equivalent for a directory of `.dat` files.

In [ ]:
from varistar import TimeSeries, LightCurve
from varistar.ml.data import Dataset
from _synthetic import make_sinusoidal_lightcurve, make_eclipsing_lightcurve

ds = Dataset(data_dir="./_scratch")  # only used if you later call export_features()

for i in range(4):
    df = make_sinusoidal_lightcurve(period=1.5 + 0.4 * i, seed=100 + i)
    ts = TimeSeries(magnitude="mag I", time_scale="HJD")
    ts.load_data_from_df(df, data_id=f"sine_{i:02d}")
    ds.add_object(ts, LightCurve(ts))

for i in range(4):
    df = make_eclipsing_lightcurve(period=3.0 + 0.5 * i, seed=200 + i)
    ts = TimeSeries(magnitude="mag I", time_scale="HJD")
    ts.load_data_from_df(df, data_id=f"eb_{i:02d}")
    ds.add_object(ts, LightCurve(ts))

len(ds)

## Feature extraction

18 statistical and astrophysical features per star (`n_cores=1` here so
docs builds stay deterministic and single-process; raise it for real
datasets):

In [ ]:
features = ds.build_features(n_cores=1)
features.head()

A single object's features, without a `Dataset`:

In [ ]:
from varistar.ml.features import FeatureExtractor

FeatureExtractor.extract(ts_obj=ts, lc_obj=LightCurve(ts))

## Clustering

In [ ]:
ds.cluster_data(n_clusters=2)
ds.visualize_clusters()

## CLI: whole directories of `.dat` files

For a real survey field, skip the Python loop and run the pipeline
directly against a directory (illustrative — not executed on this page):

```bash
python -m varistar.ml.pipeline --data ./ogle_field_01/ \
  --features 0 1 7 11 15 --output ./catalogues/
```

**Next:** [Visualization](./visualization).